# Attempt to forecast the price of MSFT by analyzing the prices of multiple stocks, including MSFT, over several consecutive days leading up to the target day.
#### N.B. Different setup from HW1

In [2]:
from torch.utils.data import DataLoader,Dataset

class StockDataset(Dataset):
    def __init__(self,X,Y,days):
        self.X = X
        self.Y = Y.reshape(-1)
        self.days = days # days ahead for prediction

    def __len__(self):
        return (len(self.Y)-self.days)

    def __getitem__(self,index):
        x=self.X[:,index:index+self.days]
        y=self.Y[index+self.days]
        return x,y



In [3]:
# !pip install pandas
# !pip install yfinance
import numpy as np
from numpy import exp, sum, log, log10
import yfinance as yf
import pandas as pd

def get_price(tick,start='2020-01-01',end=None):
    return yf.Ticker(tick).history(start=start,end=end)['Close']

def get_prices(tickers,start='2020-01-01',end=None):
    df=pd.DataFrame()
    for s in tickers:
        df[s]=get_price(s,start,end)
    return df

feature_stocks=['tsla','meta','nvda','amzn','nflx','gbtc','gdx','intc','dal','c','goog','aapl','msft','ibm','hp','orcl','sap','crm','hubs','twlo']
predict_stock='msft'

# getting data
start_date='2020-01-01'

allX=get_prices(feature_stocks,start=start_date)
ally=get_prices([predict_stock],start=start_date)

In [4]:
import torch.utils.data as data
import torch

stockData = StockDataset(allX.to_numpy().transpose().astype(np.float32),ally.to_numpy().astype(np.float32),days=5)
train_set_size = int(len(stockData)*0.7)
valid_set_size = int(len(stockData)*0.2)
test_set_size = len(stockData)-train_set_size-valid_set_size

train_set, valid_set, test_set = data.random_split(stockData,[train_set_size,valid_set_size,test_set_size],\
                                              generator=torch.Generator().manual_seed(42))

batch_size = train_set_size # use entire dataset as batch
train_dataloader = DataLoader(train_set,batch_size=batch_size,shuffle=True)  # input:(20,5), label:1
valid_dataloader = DataLoader(valid_set,batch_size=batch_size,shuffle=False)
test_dataloader = DataLoader(test_set,batch_size=batch_size,shuffle=False)

# 1. Build a simple MLP to forecast MSFT price using PyTorch Lightning.

#### You have total freedom of your MLP. But your MLP should take the last five day ($5 \times 20=100$) prices as input and you have to add dropout into your network.

## 1a. Create a subclass of pytorch_lightning.LightningModule. It should include \_\_init\_\_, training_step, validation_step, configure_optimizers in the class. (6 points)

## 1b. Create a subclass of pytorch_lightning.LightningDataModule. It should include \_\_init\_\_, train_dataloader, and val_dataloader in the class. (4 points)

## 1c. Complete the rest of the code and train the model with 70% of the data. You should set aside 15% of the data each for validation and testing.  Show the training and validation MSE (5 points)

# 2. Construct a 1-D CNN to forecast MSFT stock price. You are free to use any design, but your network must consist of at least one convolutional layer and one dropout layer. You can also extend the duration leading up to the target day by modifying the "days" argument in the StockDataset. But "days" should not be larger than 32. (10 points)


# 3. Please try to enhance the performance of the previously created MLP or CNN by applying hyperparameter tuning. You can use tools such as W&B hyperparameter sweep, SMAP, Optuna, or similar packages to achieve this. You need to optimize at least two parameters, with the dropout rate being one of them. (5 points)












In [5]:
!pip install pytorch_lightning
import numpy as np
import pandas as pd
import yfinance as yf
import torch
from torch.utils.data import Dataset, DataLoader
import torch.nn as nn
import pytorch_lightning as pl

# =========================
# 1. Dataset
# =========================
class StockDataset(Dataset):
    def __init__(self, X, Y, days=5):
        """
        X: shape [num_features, num_time_steps]
        Y: shape [num_time_steps] or [num_time_steps, 1]
        """
        self.X = X.astype(np.float32)
        self.Y = Y.reshape(-1).astype(np.float32)
        self.days = days

    def __len__(self):
        return len(self.Y) - self.days

    def __getitem__(self, index):
        # x shape: [num_features, days]
        x = self.X[:, index:index + self.days]
        y = self.Y[index + self.days]

        # flatten to 100 = 20 * 5
        x = torch.tensor(x, dtype=torch.float32).reshape(-1)
        y = torch.tensor(y, dtype=torch.float32)

        return x, y


# =========================
# 2. Data download functions
# =========================
def get_price(tick, start='2020-01-01', end=None):
    return yf.Ticker(tick).history(start=start, end=end)['Close']

def get_prices(tickers, start='2020-01-01', end=None):
    df = pd.DataFrame()
    for s in tickers:
        df[s] = get_price(s, start, end)
    return df


# =========================
# 3. Lightning DataModule
# =========================
class StockDataModule(pl.LightningDataModule):
    def __init__(self, feature_stocks, predict_stock, start_date='2020-01-01',
                 days=5, batch_size=32):
        super().__init__()
        self.feature_stocks = feature_stocks
        self.predict_stock = predict_stock
        self.start_date = start_date
        self.days = days
        self.batch_size = batch_size

    def setup(self, stage=None):
        allX = get_prices(self.feature_stocks, start=self.start_date)
        ally = get_prices([self.predict_stock], start=self.start_date)

        # align dates and drop missing rows
        data = allX.copy()
        data['target'] = ally[self.predict_stock]
        data = data.dropna()

        X = data[self.feature_stocks].to_numpy().T   # [20, T]
        Y = data['target'].to_numpy()                # [T]

        full_dataset = StockDataset(X, Y, days=self.days)

        n = len(full_dataset)
        train_size = int(n * 0.70)
        val_size = int(n * 0.15)
        test_size = n - train_size - val_size

        # time-order split, not random split
        self.train_dataset = torch.utils.data.Subset(full_dataset, range(0, train_size))
        self.val_dataset = torch.utils.data.Subset(full_dataset, range(train_size, train_size + val_size))
        self.test_dataset = torch.utils.data.Subset(full_dataset, range(train_size + val_size, n))

    def train_dataloader(self):
        return DataLoader(self.train_dataset, batch_size=self.batch_size, shuffle=False)

    def val_dataloader(self):
        return DataLoader(self.val_dataset, batch_size=self.batch_size, shuffle=False)

    def test_dataloader(self):
        return DataLoader(self.test_dataset, batch_size=self.batch_size, shuffle=False)


# =========================
# 4. Lightning Module
# =========================
class MLPForecaster(pl.LightningModule):
    def __init__(self, input_dim=100, hidden_dim1=128, hidden_dim2=64,
                 dropout=0.3, lr=1e-3):
        super().__init__()
        self.save_hyperparameters()

        self.model = nn.Sequential(
            nn.Linear(input_dim, hidden_dim1),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim1, hidden_dim2),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim2, 1)
        )

        self.criterion = nn.MSELoss()

    def forward(self, x):
        return self.model(x).squeeze(-1)

    def training_step(self, batch, batch_idx):
        x, y = batch
        y_hat = self(x)
        loss = self.criterion(y_hat, y)
        self.log("train_mse", loss, prog_bar=True, on_epoch=True, on_step=False)
        return loss

    def validation_step(self, batch, batch_idx):
        x, y = batch
        y_hat = self(x)
        val_loss = self.criterion(y_hat, y)
        self.log("val_mse", val_loss, prog_bar=True, on_epoch=True, on_step=False)
        return val_loss

    def configure_optimizers(self):
        return torch.optim.Adam(self.parameters(), lr=self.hparams.lr)


# =========================
# 5. Run training
# =========================
feature_stocks = [
    'tsla', 'meta', 'nvda', 'amzn', 'nflx',
    'gbtc', 'gdx', 'intc', 'dal', 'c',
    'goog', 'aapl', 'msft', 'ibm', 'hpq',
    'orcl', 'sap', 'crm', 'hubs', 'twlo'
]
predict_stock = 'msft'

dm = StockDataModule(
    feature_stocks=feature_stocks,
    predict_stock=predict_stock,
    start_date='2020-01-01',
    days=5,
    batch_size=32
)

model = MLPForecaster(
    input_dim=5 * 20,
    hidden_dim1=128,
    hidden_dim2=64,
    dropout=0.3,
    lr=1e-3
)

trainer = pl.Trainer(
    max_epochs=50,
    enable_checkpointing=False,
    logger=False
)

trainer.fit(model, datamodule=dm)

# =========================
# 6. Show training and validation MSE
# =========================
print("Final training metrics:")
print(trainer.callback_metrics)

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 857.3/857.3 kB 13.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 983.4/983.4 kB 39.9 MB/s eta 0:00:00


INFO:pytorch_lightning.utilities.rank_zero:GPU available: False, used: False
INFO:pytorch_lightning.utilities.rank_zero:TPU available: False, using: 0 TPU cores
INFO:pytorch_lightning.utilities.rank_zero:💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.


┏━━━┳━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━┳━━━━━━━┳━━━━━━━┓
┃   ┃ Name      ┃ Type       ┃ Params ┃ Mode  ┃ FLOPs ┃
┡━━━╇━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━╇━━━━━━━╇━━━━━━━┩
│ 0 │ model     │ Sequential │ 21.2 K │ train │     0 │
│ 1 │ criterion │ MSELoss    │      0 │ train │     0 │
└───┴───────────┴────────────┴────────┴───────┴───────┘

Trainable params: 21.2 K                                                                                           
Non-trainable params: 0                                                                                            
Total params: 21.2 K                                                                                               
Total estimated model params size (MB): 0                                                                          
Modules in train mode: 9                                                                                           
Modules in eval mode: 0                                                                                            
Total FLOPs: 0

Output()

/usr/local/lib/python3.12/dist-packages/pytorch_lightning/utilities/_pytree.py:21: `isinstance(treespec, LeafSpec)`
is deprecated, use `isinstance(treespec, TreeSpec) and treespec.is_leaf()` instead.

INFO:pytorch_lightning.utilities.rank_zero:`Trainer.fit` stopped: `max_epochs=50` reached.


Final training metrics:
{'val_mse': tensor(2019.2904), 'train_mse': tensor(2570.8896)}
